# Structured logging

Stable fields make logs searchable and connect events from one request.

In [ ]:
import json

event = {
    "event": "request_finished",
    "request_id": "req-123",
    "status_code": 200,
    "duration_ms": 24,
}

print(json.dumps(event))

## Polished version

A standalone standard-library formatter emits one JSON event with stable fields and request context, matching production log collection.

In [ ]:
import json
import logging
from datetime import datetime, timezone
from io import StringIO


class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        payload = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        for field in ("request_id", "method", "path", "status_code", "duration_ms"):
            if hasattr(record, field):
                payload[field] = getattr(record, field)
        if record.exc_info:
            payload["exception"] = self.formatException(record.exc_info)
        return json.dumps(payload, sort_keys=True)


stream = StringIO()
handler = logging.StreamHandler(stream)
handler.setFormatter(JsonFormatter())

logger = logging.getLogger("demo.http")
logger.handlers = [handler]
logger.setLevel(logging.INFO)
logger.propagate = False

logger.info(
    "request_completed",
    extra={
        "request_id": "req-456",
        "method": "POST",
        "path": "/v1/tasks",
        "status_code": 201,
        "duration_ms": 18.4,
    },
)

print(json.loads(stream.getvalue()))

## Applied in this repository

Both projects use this formatter-and-request-context pattern in [REST logging.py](../00P1-project-rest-api/app/logging.py) and [LLM logging.py](../00P2-project-llm-api/app/logging.py).